<a href="https://colab.research.google.com/github/Rumeysalle/Skin-Cancer-Analysis-/blob/main/Ham10000.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%matplotlib inline
import os
import cv2
import itertools
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
from glob import glob
from PIL import Image

# PyTorch ve Görselleştirme
import torch
from torch import optim, nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

# Sklearn (Model Değerlendirme ve Veri Bölme)
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

np.random.seed(10)
torch.manual_seed(10)
torch.cuda.manual_seed(10)


print(os.listdir("/content/drive/MyDrive/HAM10000/all_images"))

['hmnist_28_28_L.csv', 'HAM10000_metadata.csv', 'hmnist_8_8_RGB.csv', 'hmnist_8_8_L.csv', 'hmnist_28_28_RGB.csv', 'HAM10000_images_part_2', 'HAM10000_images_part_1']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/ham10000.zip /content/
!unzip -q /content/ham10000.zip -d /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data_dir = '/content/ham10000_zip'

all_image_path = glob(os.path.join(data_dir, '*', '*.jpg'))
print(f"Toplam görüntü sayısı: {len(all_image_path)}")  # 10015


imageid_path_dict = {
    os.path.splitext(os.path.basename(x))[0]: x
    for x in all_image_path
}

lesion_type_dict = {
    'nv':    'Melanocytic nevi',
    'mel':   'Melanoma',
    'bkl':   'Benign keratosis-like lesions',
    'bcc':   'Basal cell carcinoma',
    'akiec': 'Actinic keratoses',
    'vasc':  'Vascular lesions',
    'df':    'Dermatofibroma'
}

df = pd.read_csv(os.path.join(data_dir, 'HAM10000_metadata.csv'))
df['path'] = df['image_id'].map(imageid_path_dict)
df['cell_type'] = df['dx'].map(lesion_type_dict)

print(df.shape)
print(df.head())

Toplam görüntü sayısı: 10015
(10015, 9)
     lesion_id      image_id   dx dx_type   age   sex localization  \
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp   
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp   
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp   
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp   
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear   

                                                path  \
0  /content/ham10000_zip/HAM10000_images_part_1/I...   
1  /content/ham10000_zip/HAM10000_images_part_1/I...   
2  /content/ham10000_zip/HAM10000_images_part_1/I...   
3  /content/ham10000_zip/HAM10000_images_part_1/I...   
4  /content/ham10000_zip/HAM10000_images_part_2/I...   

                       cell_type  
0  Benign keratosis-like lesions  
1  Benign keratosis-like lesions  
2  Benign keratosis-like lesions  
3  Benign keratosis-like lesions  
4  Benign keratosis-like le

In [ ]:
def compute_img_mean_std_batch(image_paths, batch_size=500):
    img_h, img_w = 224, 224

    channel_sum = np.zeros(3)
    channel_sum_sq = np.zeros(3)
    total_pixels = 0

    for i in tqdm(range(0, len(image_paths), batch_size)):
        batch_paths = image_paths[i:i+batch_size]

        for path in batch_paths:
            img = cv2.imread(path)
            img = cv2.resize(img, (img_h, img_w))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # BGR → RGB
            img = img.astype(np.float32) / 255.

            channel_sum += img.sum(axis=(0,1))
            channel_sum_sq += (img**2).sum(axis=(0,1))
            total_pixels += img_h * img_w

    mean = channel_sum / total_pixels
    std = np.sqrt(channel_sum_sq/total_pixels - mean**2)

    print(f"normMean = {mean.tolist()}")
    print(f"normStd  = {std.tolist()}")
    return mean.tolist(), std.tolist()

norm_mean, norm_std = compute_img_mean_std_batch(all_image_path)

100%|██████████| 21/21 [01:50<00:00,  5.26s/it]

normMean = [0.763153348547253, 0.5455634293418283, 0.5699802738108737]
normStd  = [0.1402666239351315, 0.1528786665257668, 0.1701613301374165]


In [ ]:
df_original = pd.read_csv(os.path.join(data_dir, 'HAM10000_metadata.csv'))
df_original['path'] = df_original['image_id'].map(imageid_path_dict.get)
df_original['cell_type'] = df_original['dx'].map(lesion_type_dict.get)
df_original['cell_type_idx'] = pd.Categorical(df_original['cell_type']).codes
df_original.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path,cell_type,cell_type_idx
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,/content/ham10000_zip/HAM10000_images_part_1/I...,Benign keratosis-like lesions,2
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,/content/ham10000_zip/HAM10000_images_part_1/I...,Benign keratosis-like lesions,2
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,/content/ham10000_zip/HAM10000_images_part_1/I...,Benign keratosis-like lesions,2
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,/content/ham10000_zip/HAM10000_images_part_1/I...,Benign keratosis-like lesions,2
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,/content/ham10000_zip/HAM10000_images_part_2/I...,Benign keratosis-like lesions,2


In [ ]:
# this will tell us how many images are associated with each lesion_id
df_undup = df_original.groupby('lesion_id').count()
# now we filter out lesion_id's that have only one image associated with it
df_undup = df_undup[df_undup['image_id'] == 1]
df_undup.reset_index(inplace=True)
df_undup.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path,cell_type,cell_type_idx
0,HAM_0000001,1,1,1,1,1,1,1,1,1
1,HAM_0000003,1,1,1,1,1,1,1,1,1
2,HAM_0000004,1,1,1,1,1,1,1,1,1
3,HAM_0000007,1,1,1,1,1,1,1,1,1
4,HAM_0000008,1,1,1,1,1,1,1,1,1


In [ ]:
def get_duplicates(x):
    unique_list = list(df_undup['lesion_id'])
    if x in unique_list:
        return 'unduplicated'
    else:
        return 'duplicated'

# create a new column that is a copy of the lesion_id column
df_original['duplicates'] = df_original['lesion_id']
# apply the function to this new column
df_original['duplicates'] = df_original['duplicates'].apply(get_duplicates)
df_original.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path,cell_type,cell_type_idx,duplicates
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,/content/ham10000_zip/HAM10000_images_part_1/I...,Benign keratosis-like lesions,2,duplicated
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,/content/ham10000_zip/HAM10000_images_part_1/I...,Benign keratosis-like lesions,2,duplicated
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,/content/ham10000_zip/HAM10000_images_part_1/I...,Benign keratosis-like lesions,2,duplicated
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,/content/ham10000_zip/HAM10000_images_part_1/I...,Benign keratosis-like lesions,2,duplicated
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,/content/ham10000_zip/HAM10000_images_part_2/I...,Benign keratosis-like lesions,2,duplicated


In [ ]:
df_original['duplicates'].value_counts()

,count
duplicates,
unduplicated,5514
duplicated,4501


In [ ]:
# now we filter out images that don't have duplicates
df_undup = df_original[df_original['duplicates'] == 'unduplicated']
df_undup.shape

(5514, 11)

In [ ]:
y = df_undup['cell_type_idx']
_, df_val = train_test_split(df_undup, test_size=0.2, random_state=101, stratify=y)
df_val.shape

(1103, 11)

In [ ]:
df_val['cell_type_idx'].value_counts()

,count
cell_type_idx,
4,883
2,88
5,46
1,35
0,30
6,13
3,8


In [ ]:
# This set will be df_original excluding all rows that are in the val set
# This function identifies if an image is part of the train or val set.
def get_val_rows(x):
    # create a list of all the lesion_id's in the val set
    val_list = list(df_val['image_id'])
    if str(x) in val_list:
        return 'val'
    else:
        return 'train'

# identify train and val rows
# create a new colum that is a copy of the image_id column
df_original['train_or_val'] = df_original['image_id']
# apply the function to this new column
df_original['train_or_val'] = df_original['train_or_val'].apply(get_val_rows)
# filter out train rows
df_train = df_original[df_original['train_or_val'] == 'train']
print(len(df_train))
print(len(df_val))

8912
1103


In [ ]:
df_train['cell_type_idx'].value_counts()

,count
cell_type_idx,
4,5822
5,1067
2,1011
1,479
0,297
6,129
3,107


In [ ]:
df_val['cell_type'].value_counts()

,count
cell_type,
Melanocytic nevi,883
Benign keratosis-like lesions,88
Melanoma,46
Basal cell carcinoma,35
Actinic keratoses,30
Vascular lesions,13
Dermatofibroma,8
